# Producer and Consumer with `asyncio`

In this chapter, we introduce the **producer–consumer pattern** using Python’s asynchronous programming model. A **producer** generates events (data), while a **consumer** processes them as they become available. Both components work together through a **queue**, without blocking each other.

You will build a small producer-consumer example with `asyncio.Queue`. It keeps the streaming setup local while showing the same ideas you would later use with a broker such as Redpanda.

**Producer:**
- Generates data
- `queue.put()` → pushes items to the queue

**Consumer:**
- Waits for data
- `queue.get()` → processes items from the queue

```mermaid
flowchart LR
    P["Producer(s)"] --> Q["asyncio.Queue"]
    Q --> C["Consumer(s)"]
```
**Learning goals:**
- Model events with a small Python object.
- Push events into an `asyncio.Queue`.
- Consume events, classify them, and print window summaries.


## Outline

1. Define the event shape.
2. Create a producer and a consumer.
3. Run both tasks together.
4. Adjust the window size and threshold.


In [ ]:
import asyncio
from dataclasses import dataclass


@dataclass(slots=True)
class TaxiRide:
    vendor_id: int
    total_amount: float
    trip_distance: float


sample_rides = [
    TaxiRide(1, 14.5, 1.2),
    TaxiRide(2, 47.2, 8.4),
    TaxiRide(1, 18.8, 2.3),
    TaxiRide(2, 63.4, 12.7),
    TaxiRide(1, 29.1, 4.9),
]

## Step 1 - Define the async workers

The producer adds rides to the queue. The consumer reads them, classifies them, and prints a summary every few events.


In [ ]:
async def producer(queue, rides, delay_seconds=0.05):
    for ride in rides:
        print(f"Produced {ride}")
        await queue.put(ride)
        await asyncio.sleep(delay_seconds)
    await queue.put(None)


def summarize(window, threshold):
    return {
        "events": len(window),
        "high_amount": sum(ride.total_amount >= threshold for ride in window),
        "low_amount": sum(ride.total_amount < threshold for ride in window),
        "avg_total_amount": round(
            sum(ride.total_amount for ride in window) / len(window), 2
        ),
    }


async def consumer(queue, window_size=2, threshold=40.0):
    window = []
    while True:
        ride = await queue.get()
        if ride is None:
            if window:
                print("Final window:", summarize(window, threshold))
            return

        bucket = "high_amount" if ride.total_amount >= threshold else "low_amount"
        print(f"Consumed {ride.vendor_id=} -> {bucket}")
        window.append(ride)

        if len(window) == window_size:
            print("Window summary:", summarize(window, threshold), "\n")
            window = []

## Step 2 - Run the producer and consumer together

Jupyter supports top-level `await`, so we can run the demo directly in the notebook.


In [ ]:
async def run_demo(window_size=2, threshold=40.0):
    queue = asyncio.Queue()
    await asyncio.gather(
        producer(queue, sample_rides),
        consumer(queue, window_size=window_size, threshold=threshold),
    )


await run_demo(window_size=2, threshold=40.0)

## Step 3 - Try a different configuration

Small parameter changes help you see how windows and thresholds change the output story.


In [ ]:
await run_demo(window_size=3, threshold=30.0)

## Exercises

- Modify the `summarize()` and `consumer()` functions to classify rides into three categories (economy, standard and premium).
- Create a consumer function that prints the most expensive ride for each window.
- Explain how this notebook would change if the queue were replaced by a real broker.


> **Common pitfall:** In a notebook, prefer top-level `await` over `asyncio.run(...)` because Jupyter already manages an event loop.

In [ ]:
# TODO: Write your solutions for the exercises above.


<details>
    <summary>Show Reference Solution</summary>

```python
def summarize_3_categories(window, economy_limit=25, standard_limit=50):
    
    economy = sum(ride.total_amount < economy_limit for ride in window)
    standard = sum(economy_limit <= ride.total_amount <= standard_limit for ride in window)
    premium = sum(ride.total_amount > standard_limit for ride in window)
    
    return {
        "events": len(window),
        "economy": economy,
        "standard": standard,
        "premium": premium,
        "avg_total_amount": round(
            sum(ride.total_amount for ride in window) / len(window), 2
        ),
    }


async def consumer_3_categories(queue, window_size=2, economy_limit=25, standard_limit=50):
    window = []
    while True:
        ride = await queue.get()
        if ride is None:
            if window:
                print("Final window:", summarize_3_categories(window, economy_limit, standard_limit))
            return

        if ride.total_amount < economy_limit:
            category = "economy"
        elif ride.total_amount <= standard_limit:
            category = "standard"
        else:
            category = "premium"
        
        print(f"Consumed {ride.vendor_id=} -> {category}")
        window.append(ride)

        if len(window) == window_size:
            print("Window summary:", summarize_3_categories(window, economy_limit, standard_limit),"\n")
            window = []


async def run_demo_3_categories(window_size=2, economy_limit=25, standard_limit=50):
    queue = asyncio.Queue()
    await asyncio.gather(
        producer(queue, sample_rides),
        consumer_3_categories(queue, window_size=window_size, economy_limit=economy_limit, standard_limit=standard_limit),
    )


await run_demo_3_categories(window_size=3, economy_limit=25, standard_limit=50)

# ----------

async def consumer_max_ride(queue, window_size=2):
    window = []
    while True:
        ride = await queue.get()
        if ride is None:
            if window:
                max_ride = max(window, key=lambda r: r.total_amount)
                print(f"Final window: {len(window)} rides - Most expensive ride: ${max_ride.total_amount}")
            return
        
        window.append(ride)
        
        if len(window) == window_size:
            max_ride = max(window, key=lambda r: r.total_amount)            
            print(f"Window: {len(window)} rides - Most expensive ride: ${max_ride.total_amount}\n")
            window = []


async def run_demo_max(window_size=2):
    queue = asyncio.Queue()
    await asyncio.gather(
        producer(queue, sample_rides),
        consumer_max_ride(queue, window_size=window_size),
    )


await run_demo_max(window_size=3)
```
</details>
